# Business Data Practice: CSV and Excel with Python
This notebook uses sample sales data. Run cells from top to bottom. It demonstrates reading CSV, writing CSV, reading Excel, analyzing data, and generating an Excel report.

**Online option:** upload `sample_sales.csv` to Google Colab, or create it using the first code cell below.

## 0. Setup and create sample data
Run this cell to create a small input CSV so no separate download is required.

In [ ]:
import pandas as pd
from pathlib import Path

data = [
    {'Date': '2026-10-01', 'Employee': 'Ayesha', 'Department': 'Sales', 'Product': 'Laptop', 'Quantity': 2, 'UnitPrice': 85000},
    {'Date': '2026-10-02', 'Employee': 'Bilal', 'Department': 'Sales', 'Product': 'Monitor', 'Quantity': 4, 'UnitPrice': 32000},
    {'Date': '2026-10-03', 'Employee': 'Hina', 'Department': 'Finance', 'Product': 'Keyboard', 'Quantity': 10, 'UnitPrice': 3500},
    {'Date': '2026-10-04', 'Employee': 'Usman', 'Department': 'Sales', 'Product': 'Laptop', 'Quantity': 1, 'UnitPrice': 85000},
    {'Date': '2026-10-05', 'Employee': 'Sara', 'Department': 'IT', 'Product': 'Monitor', 'Quantity': 3, 'UnitPrice': 32000},
    {'Date': '2026-10-06', 'Employee': 'Hamza', 'Department': 'Finance', 'Product': 'Keyboard', 'Quantity': 6, 'UnitPrice': 3500},
    {'Date': '2026-10-07', 'Employee': 'Mariam', 'Department': 'Sales', 'Product': 'Mouse', 'Quantity': 8, 'UnitPrice': 1800},
    {'Date': '2026-10-08', 'Employee': 'Danish', 'Department': 'IT', 'Product': 'Laptop', 'Quantity': 1, 'UnitPrice': 85000},
]
df = pd.DataFrame(data)
df.to_csv('sample_sales.csv', index=False)
print('Created sample_sales.csv')
df.head()

## 1. Read a CSV, calculate revenue, and generate a new CSV
Business task: calculate revenue for each transaction and save transactions worth at least 50,000.

In [ ]:
sales = pd.read_csv('sample_sales.csv')
sales['Revenue'] = sales['Quantity'] * sales['UnitPrice']
high_value_sales = sales[sales['Revenue'] >= 50000]
high_value_sales.to_csv('high_value_sales.csv', index=False)
print('Transactions:', len(sales))
print('High-value transactions:', len(high_value_sales))
high_value_sales[['Employee', 'Product', 'Revenue']]

## 2. Generate an Excel workbook
Business task: create an Excel file with detailed sales, product totals, and department totals.

In [ ]:
sales['Revenue'] = sales['Quantity'] * sales['UnitPrice']
product_summary = sales.groupby('Product', as_index=False).agg(
    UnitsSold=('Quantity', 'sum'), Revenue=('Revenue', 'sum')
).sort_values('Revenue', ascending=False)
department_summary = sales.groupby('Department', as_index=False).agg(
    Transactions=('Product', 'count'), Revenue=('Revenue', 'sum')
).sort_values('Revenue', ascending=False)

with pd.ExcelWriter('sales_analysis_report.xlsx', engine='openpyxl') as writer:
    sales.to_excel(writer, sheet_name='Sales Data', index=False)
    product_summary.to_excel(writer, sheet_name='Product Summary', index=False)
    department_summary.to_excel(writer, sheet_name='Department Summary', index=False)

print('Created sales_analysis_report.xlsx')
product_summary

## 3. Read an Excel file and answer business questions
Business task: reopen the workbook, identify the top product, and find the department with the highest revenue.

In [ ]:
report = pd.read_excel('sales_analysis_report.xlsx', sheet_name='Product Summary')
top_product = report.sort_values('Revenue', ascending=False).iloc[0]
print(f"Top product by revenue: {top_product['Product']} (revenue: {top_product['Revenue']:,.0f})")

department_report = pd.read_excel('sales_analysis_report.xlsx', sheet_name='Department Summary')
top_department = department_report.sort_values('Revenue', ascending=False).iloc[0]
print(f"Top department by revenue: {top_department['Department']} (revenue: {top_department['Revenue']:,.0f})")
department_report

## 4. Mini-project: create a reusable report function
Business task: run the same process for another CSV later by reusing a function.

In [ ]:
def create_sales_report(input_csv, output_excel, output_summary_csv):
    sales = pd.read_csv(input_csv)
    sales['Revenue'] = sales['Quantity'] * sales['UnitPrice']
    summary = sales.groupby('Product', as_index=False).agg(
        UnitsSold=('Quantity', 'sum'), Revenue=('Revenue', 'sum')
    ).sort_values('Revenue', ascending=False)
    with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
        sales.to_excel(writer, sheet_name='Sales Data', index=False)
        summary.to_excel(writer, sheet_name='Product Summary', index=False)
    summary.to_csv(output_summary_csv, index=False)
    return summary

final_summary = create_sales_report(
    'sample_sales.csv', 'final_sales_report.xlsx', 'product_summary.csv'
)
print('Created final_sales_report.xlsx and product_summary.csv')
final_summary

## Practice challenges
1. Change the high-value threshold from 50,000 to 100,000.
2. Add a `Discount` column and calculate `NetRevenue`.
3. Find the employee with the highest total revenue.
4. Add a new worksheet showing only Sales department transactions.
5. Save the final notebook and generated files in a GitHub repository.